In [ ]:
#  ALLIGATOR SEGMENTATION (Frames 15-30)
import cv2
import numpy as np
import os

# ====== INPUT FOLDER ======
input_folder = "road_member3_sharpened"

# ====== OUTPUT FOLDERS - one for each stage ======
folders = {
    "input":     "output_member2/1_input_frames",
    "gray":      "output_member2/2_gray_frames",
    "blur":      "output_member2/3_blur_frames",
    "blackhat":  "output_member2/4_blackhat_frames",
    "thresh":    "output_member2/5_thresh_frames",
    "opening":   "output_member2/6_opening_frames",
    "dilation":  "output_member2/7_dilation_frames",
    "edges":     "output_member2/8_edges_frames",
    "contours":  "output_member2/9_contours_frames",
}

# Create all folders
for folder in folders.values():
    os.makedirs(folder, exist_ok=True)

# ====== PROCESS FRAMES 17 TO 30 ======
for i in range(17, 31):

    # INPUT FRAME
    img_path = os.path.join(input_folder, f"gaussian_contrast_frame_{i}.jpg")
    image = cv2.imread(img_path)

    if image is None:
        print(f"Frame {i} not found, skipping...")
        continue

    # Save input
    cv2.imwrite(os.path.join(folders["input"], f"frame_{i}.jpg"), image)

    # PREPROCESSING - GRAYSCALE
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    cv2.imwrite(os.path.join(folders["gray"], f"frame_{i}.jpg"), gray)

    # PREPROCESSING - GAUSSIAN BLUR
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    cv2.imwrite(os.path.join(folders["blur"], f"frame_{i}.jpg"), blur)

    # SEGMENTATION - BLACKHAT
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
    blackhat = cv2.morphologyEx(blur, cv2.MORPH_BLACKHAT, kernel)
    cv2.imwrite(os.path.join(folders["blackhat"], f"frame_{i}.jpg"), blackhat)

    # SEGMENTATION - THRESHOLD (Otsu for better accuracy)
    norm = cv2.normalize(blackhat, None, 0, 255, cv2.NORM_MINMAX)
    _, thresh = cv2.threshold(norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    cv2.imwrite(os.path.join(folders["thresh"], f"frame_{i}.jpg"), thresh)

    # MORPHOLOGICAL CLEANING - OPENING
    kernel_small = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel_small)
    cv2.imwrite(os.path.join(folders["opening"], f"frame_{i}.jpg"), opening)

    # MORPHOLOGICAL CLEANING - DILATION (used as predicted mask)
    dilation = cv2.dilate(opening, kernel_small, iterations=2)
    cv2.imwrite(os.path.join(folders["dilation"], f"frame_{i}.jpg"), dilation)

    # EDGE DETECTION
    edges = cv2.Canny(dilation, 50, 150)
    cv2.imwrite(os.path.join(folders["edges"], f"frame_{i}.jpg"), edges)

    # CONTOUR DETECTION + FILTERING
    contours, _ = cv2.findContours(dilation, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    output = image.copy()

    for cnt in contours:
        area = cv2.contourArea(cnt)

        if area > 200:
            x, y, w, h = cv2.boundingRect(cnt)
            roi = dilation[y:y+h, x:x+w]
            density = cv2.countNonZero(roi) / (w * h)

            if 0.15 < density < 0.5:
                cv2.rectangle(output, (x, y), (x+w, y+h), (0, 255, 0), 2)
                cv2.putText(output, "Alligator Crack", (x, y-5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

    cv2.imwrite(os.path.join(folders["contours"], f"frame_{i}.jpg"), output)

    print(f"Frame {i} processed and saved.")

print("\nAll frames processed successfully!")
print("\nFolder structure created:")
for name, path in folders.items():
    print(f"  {path}/")